# 11 · Graph-based representations
Hypothesis: perturbing a gene affects its network neighbours, so **graph-smoothed** gene features generalise better than raw features. We build a kNN gene graph (co-expression, or STRING if you provide an edge file), propagate embeddings over it (SGC-style), and re-run the benchmark. Section B optionally runs **GEARS**, a published GNN for perturbation prediction.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
from perturbai.pipeline import load_task
from perturbai.features.graph import knn_graph, propagate, string_adjacency
from perturbai.models.baselines import MeanBaseline, RidgeModel, KNNModel
from perturbai.models.mlp import MLPModel
from perturbai.evaluation.benchmark import run_benchmark, summarize, compare_to_baseline
T = load_task(P, TAG, ("coexp_",)); feats = T["feats"]
G = MCFG["features"]
string_file = P.raw / "string" / "string_edges_symbols.csv"     # optional: columns protein1, protein2, combined_score (gene symbols)
if string_file.exists():
    A = string_adjacency(pd.read_csv(string_file), list(feats.index), threshold=700); print("STRING graph edges:", A.nnz // 2)
else:
    A = knn_graph(feats.values, G["graph_knn"]); print("co-expression kNN graph edges:", A.nnz // 2)
Z = propagate(A, feats.values, G["graph_steps"], G["graph_alpha"])
Zdf = pd.DataFrame(Z, index=feats.index); pos = {g: i for i, g in enumerate(Zdf.index)}
Xg = Zdf.loc[T["index"]].values

In [ ]:
M = MCFG["models"]
facts = {"mean": MeanBaseline, "ridge_graph": lambda: RidgeModel(**M["ridge"]), "knn_graph": lambda: KNNModel(**M["knn"])}
long = run_benchmark(facts, Xg, T["Y"], T["index"], T["ctrl_mean"], seeds=MCFG["split"]["seeds"], test_frac=MCFG["split"]["test_frac"])
long = long[long.model != "mean"]
long.to_csv(P.results / f"{TAG}_bench_graph.csv", index=False)
base = pd.read_csv(P.results / f"{TAG}_bench_baselines.csv")
allr = pd.concat([base, long]); display(summarize(allr).round(3)); display(compare_to_baseline(allr, "mean").round(3))

## B. GEARS (optional, GPU recommended)
Install in a **fresh runtime** because of heavy dependencies: `pip install torch_geometric cell-gears` (see GEARS repo for CUDA-matched wheels). The calls below follow GEARS' documented API; **verify argument names against your installed version**, since the API has changed between releases. GEARS downloads a GO graph on first use (needs internet).

Fair comparison: take the perturbation split GEARS uses, then evaluate *our* models on exactly that split with `evaluate_on_split`.

In [ ]:
RUN_GEARS = False
if RUN_GEARS:
    import scanpy as sc
    from gears import PertData, GEARS
    a = sc.read_h5ad(P.processed / artifact("subset_adata", TAG))
    a.obs["condition"] = a.obs[PERT].astype(str).map(lambda g: "ctrl" if g == CTRL else f"{g}+ctrl")
    a.obs["cell_type"] = "K562"; a.var["gene_name"] = a.var_names
    pd_ = PertData(str(P.scratch / "gears_data"))
    pd_.new_data_process(dataset_name="subset", adata=a)
    pd_.load(data_path=str(P.scratch / "gears_data" / "subset"))
    pd_.prepare_split(split="simulation_single", seed=1)
    pd_.get_dataloader(batch_size=32, test_batch_size=128)
    gm = GEARS(pd_, device="cuda"); gm.model_initialize(hidden_size=64)
    gm.train(epochs=15, lr=1e-3)
    test_perts = [c.replace("+ctrl", "") for c in pd_.set2conditions["test"] if c != "ctrl"]
    train_perts = [c.replace("+ctrl", "") for c in pd_.set2conditions["train"] if c != "ctrl"]
    pred = gm.predict([[g] for g in test_perts])        # dict: 'GENE' -> expression vector
    ctrl_log = T["ctrl_mean"]
    gears_delta = np.vstack([np.asarray(pred[g]) - ctrl_log for g in test_perts])  # gene order must match T['genes']; check!
    from perturbai.evaluation.metrics import per_perturbation_metrics
    gears_metrics = per_perturbation_metrics(gears_delta, T["Y"][[list(T["index"]).index(g) for g in test_perts]], T["ctrl_mean"], index=test_perts).assign(model="gears")
    from perturbai.evaluation.benchmark import evaluate_on_split
    ours = evaluate_on_split({"ridge": lambda: RidgeModel(10.0), "knn": lambda: KNNModel(5), "mean": MeanBaseline},
                             T["X"], T["Y"], T["index"], train_perts, test_perts, T["ctrl_mean"])
    cmp_ = pd.concat([gears_metrics.reset_index().rename(columns={"index": "perturbation"}), ours])
    display(cmp_.groupby("model")[["pearson_delta", "mse_delta", "direction_acc_top_deg"]].mean())

**Report:** did graph smoothing help? Did GEARS beat Ridge/kNN on the same split? If not, discuss why (small subset, weak effects, feature quality, noise ceiling).